# Pipeline de Pré-Processamento — DeepLabCut vs. TopScan
**Laboratório de Memória - Instituto do Cérebro (UFRN) | Validação Técnica de Rastreamento Animal**

---

## Visão Geral

Este notebook realiza o pré-processamento dos dados brutos de rastreamento antes da análise estatística.
Ele deve ser executado **antes** do notebook `Pipeline de Análise — DeepLabCut vs. TopScan.ipynb`.

**Fluxo da pipeline:**
```
Vídeos .MPG  →  Conversão .MP4  →  Recorte por animal
     ↓
Arquivos .H5 (DLC)  →  Filtro de Likelihood (> 0.9)  →  TXT por bodypart
     ↓
TXT TopScan (bruto)  →  Remoção de -1  →  IQR + DBSCAN  →  Savitzky-Golay  →  Calibração (Regressão)  →  TXT final
```

---

## Como Executar

Execute as células **em ordem, de cima para baixo**. Cada seção é independente após o mount do Drive.

| Seção | O que faz | Precisa configurar? |
|---|---|---|
| Instalação de Dependências | Instala ffmpeg, scipy, scikit-learn | Não |
| Conversão de Vídeos | .MPG → .MP4 com ffmpeg | Sim — `input_folder`, `output_folder`, `initial` |
| Recorte de Vídeos | Divide vídeo multi-animal por quadrante | Sim — `pasta_videos` |
| Exportação H5 → TXT | Extrai coordenadas DLC com filtro de likelihood | Sim — `experimento`, `bodypart_selecionado` |
| Calibração TopScan | Calibra, filtra outliers e mescla eventos | Sim — `base_path`, `saida_path_dir` |

---

## Parâmetros que Você Deve Ajustar

```python
# --- Seção DLC ---
experimento          = "experimento"   # Prefixo dos arquivos .h5
bodypart_selecionado = 'tail_tip'      # Bodypart de interesse (ver bodypart_mapping)
LIMIAR_LIKELIHOOD    = 0.9             # Frames abaixo disso são descartados como NaN

# --- Seção TopScan ---
base_path      = "...TXT original e XLSX original/"  # Pasta com os .TXT e .XLSX brutos
saida_path_dir = "...TXT novo/"                      # Pasta de saída dos arquivos calibrados
```

> **Dica:** Nunca altere `LIMIAR_LIKELIHOOD` para valores abaixo de `0.85`.
> Predições com likelihood baixo indicam oclusão ou falha do modelo ResNet-50.

---

## Estrutura Esperada no Google Drive

```
MyDrive/
└── validacao_topscan/
    ├── DEEPLABCUT/
    │   ├── h5_validacao/          ← arquivos .h5 exportados pelo DLC
    │   └── trajetoria_txt_dlc/    ← TXTs gerados por este notebook
    └── TOPSCAN/
        ├── TXT original e XLSX original/  ← arquivos brutos do TopScan
        └── TXT novo/                      ← TXTs calibrados gerados por este notebook
```

---

## Sobre os Filtros Aplicados

- **IQR (deslocamento):** Remove saltos frame-a-frame fisicamente impossíveis. Limiar padrão: `Q3 + 2.5 * IQR`.
- **DBSCAN:** Remove clusters de predição isolados (glitches). Parâmetro `eps=15 px` — ajustar se a resolução do vídeo for muito diferente de ~40 px/cm.
- **Savitzky-Golay:** Suaviza ruído de alta frequência sem distorcer picos de velocidade. Janela padrão: `11 frames` (~0.37 s a 30 fps).
- **Regressão Huber:** Calibra TopScan → espaço DLC com intercepto (corrige translação). R² < 0.90 indica distorção de lente — reportar ao responsável pela configuração da câmera.

---

#CONEXÃO COM O SEU DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##INSTALAÇÃO DE DEPENDÊNCIAS

In [ ]:
!apt-get install -q ffmpeg
!pip install -q ffmpeg-python moviepy
!pip install -q --upgrade pandas scikit-learn scipy

##IMPORTS

In [ ]:
import os
import re
import subprocess

import cv2
import ffmpeg
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import r2_score
from sklearn.cluster import DBSCAN
from tqdm import tqdm

##CONVERSÃO DE VÍDEOS .MPG PARA .MP4 (TOPSCAN)

In [ ]:
input_folder  = '/content/drive/My Drive/TESTES_MPG'
output_folder = '/content/drive/My Drive/TESTES_MP4'
initial = ('E9L3D2TR', 'E9L3D3TR')

os.makedirs(output_folder, exist_ok=True)

def convert_videos(input_folder, output_folder):
    files = [f for f in os.listdir(input_folder)
             if f.lower().endswith('.mpg') and f.startswith(initial)]

    with tqdm(total=len(files), desc="Convertendo vídeos", unit="vídeo") as pbar:
        for filename in files:
            input_path  = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, os.path.splitext(filename)[0] + '.mp4')

            try:
                try:
                    duration = float(ffmpeg.probe(input_path)['format']['duration'])
                except ffmpeg.Error as e:
                    print(f'\nErro ao ler {filename}: {e.stderr.decode("utf-8")}')
                    pbar.update(1)
                    continue

                command = ['ffmpeg', '-i', input_path, '-r', '30', output_path]
                process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

                last_progress = -1
                while True:
                    line = process.stderr.readline()
                    if not line and process.poll() is not None:
                        break
                    if "time=" in line:
                        match = re.search(r'time=(\d+:\d+:\d+\.\d+)', line)
                        if match:
                            current_time = sum(float(x) * 60 ** i for i, x in enumerate(reversed(match.group(1).split(':'))))
                            progress = (current_time / duration) * 100
                            if int(progress) != last_progress:
                                print(f'\rConvertendo {filename}: {int(progress)}% concluído', end='')
                                last_progress = int(progress)

                process.wait()
                print(f'\nConvertido {filename}')
                pbar.update(1)

            except Exception as e:
                print(f'\nErro ao converter {filename}: {e}\n')
                pbar.update(1)

convert_videos(input_folder, output_folder)
print("Conversão concluída!")

Convertendo vídeos:   0%|          | 0/2 [00:00<?, ?vídeo/s]

Convertendo E9L3D3TR.MPG: 99% concluído

Convertendo vídeos:  50%|█████     | 1/2 [3:20:44<3:20:44, 12044.36s/vídeo]


Convertido E9L3D3TR.MPG para E9L3D3TR.mp4

Convertendo E9L3D2TR.MPG: 99% concluído

Convertendo vídeos: 100%|██████████| 2/2 [8:31:10<00:00, 15335.15s/vídeo]


Convertido E9L3D2TR.MPG para E9L3D2TR.mp4

Conversão concluída!


##RECORTE DE VÍDEOS MULTI-ANIMAL

In [ ]:
def cortar_e_salvar_com_barra(video_path, saida_base, largura_barra=35):
    nome_arquivo      = os.path.basename(video_path)
    nome_sem_extensao = os.path.splitext(nome_arquivo)[0]
    ids               = nome_sem_extensao.split("-")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Erro ao abrir: {video_path}")
        return

    fps          = cap.get(cv2.CAP_PROP_FPS)
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    codec        = cv2.VideoWriter_fourcc(*'mp4v')

    coords = [
        (35, 0,                   frame_width // 2 + 5,  frame_height // 2),
        (frame_width // 2 + 30,   0,                      frame_width - 20,  frame_height // 2),
        (35, frame_height // 2 + 5, frame_width // 2 + 5, frame_height - 10),
    ]

    saidas = {}
    for idx in range(min(len(ids), len(coords))):
        x1, y1, x2, y2 = coords[idx]
        saida_path      = os.path.join(saida_base, f"{ids[idx].strip()}.mp4")
        saidas[idx]     = cv2.VideoWriter(saida_path, codec, fps, (x2 - x1, y2 - y1))

    cor_barra = (57, 60, 60)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        for idx, writer in saidas.items():
            x1, y1, x2, y2 = coords[idx]
            crop = frame[y1:y2, x1:x2].copy()
            crop[:, -largura_barra:] = cor_barra
            writer.write(crop)

    cap.release()
    for writer in saidas.values():
        writer.release()
    print(f"Finalizado: {nome_arquivo} -> {len(saidas)} vídeos gerados.")

pasta_videos = "/content/drive/MyDrive/videos_raquel"
saida_base   = os.path.join(pasta_videos, "videos_processados")
os.makedirs(saida_base, exist_ok=True)

for arquivo in os.listdir(pasta_videos):
    if arquivo.endswith(".mp4") and "-" in arquivo:
        cortar_e_salvar_com_barra(os.path.join(pasta_videos, arquivo), saida_base)

✔️ Finalizado: 1-2.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 3-4.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 5-6.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 7-8.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 9-10.mp4 -> Gerados 2 vídeos.
✔️ Finalizado: 11-12.mp4 -> Gerados 2 vídeos.


##EXPORTAÇÃO DOS DADOS H5 --> TXT (DEEPLABCUT) COM FILTRO DE LIKELIHOOD

In [ ]:
# =================================================================
# CONFIGURAÇÕES DO USUÁRIO
# =================================================================
experimento          = "experimento"
bodypart_selecionado = 'tail_tip'   # MEXA NESSE
LIMIAR_LIKELIHOOD    = 0.9          # frames abaixo disso viram NaN
# =================================================================

bodypart_mapping = {
    'nose': 'nose', 'neck': 'neck', 'centerbody': 'centerbody',
    'tail_base': 'tail_base', 'tail_middle': 'tail_middle', 'tail_tip': 'tail_tip',
}

diretorio_h5el = '/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/h5_validacao'
diretorio_txt  = '/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/'
os.makedirs(diretorio_txt, exist_ok=True)

def extrair_rats_validos(nome_arquivo):
    match = re.search(r'experimento(.*?)DLC', nome_arquivo, re.IGNORECASE)
    return re.findall(r'\d+', match.group(1))[:1] if match else []

for arquivo in [f for f in os.listdir(diretorio_h5el) if f.lower().startswith(experimento) and f.lower().endswith(".h5")]:
    caminho_h5 = os.path.join(diretorio_h5el, arquivo)
    rat_ids    = extrair_rats_validos(arquivo)
    print(f"\nProcessando: {arquivo} | ID: {rat_ids}")

    try:
        df          = pd.read_hdf(caminho_h5, header=[0, 1])
        num_frames  = len(df)
        coluna_base = df.columns.levels[0][0]
        bodypart_nome = bodypart_mapping.get(bodypart_selecionado, bodypart_selecionado)

        x          = df[coluna_base][bodypart_selecionado]['x']
        y          = df[coluna_base][bodypart_selecionado]['y']
        likelihood = df[coluna_base][bodypart_selecionado]['likelihood']

        # Filtro de confiança: coordenadas abaixo do limiar viram NaN
        mascara_baixa_conf = likelihood < LIMIAR_LIKELIHOOD
        x = x.where(~mascara_baixa_conf, other=np.nan)
        y = y.where(~mascara_baixa_conf, other=np.nan)

        n_rej = int(mascara_baixa_conf.sum())
        print(f"  Likelihood < {LIMIAR_LIKELIHOOD}: {n_rej} frames rejeitados ({n_rej/num_frames*100:.1f}%)")
        print(f"  Frames válidos exportados: {num_frames - n_rej}")

        rat_id   = rat_ids[0] if rat_ids else "X"
        nome_txt = f"{experimento}_R{rat_id}_trajetoria_{bodypart_nome}.txt"

        pd.DataFrame({
            'Frame': range(num_frames),
            'X': x.round(2),
            'Y': y.round(2),
        }).to_csv(
            os.path.join(diretorio_txt, nome_txt),
            sep=' ', index=False, float_format='%.2f', na_rep='NaN',
        )
        print(f"  {nome_txt} salvo.")

    except Exception as e:
        print(f"Erro no processamento: {e}")

print("\nExportação finalizada!")


Processando: experimento11DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['11']
  ✅ experimento_R11_trajetoria_tail_tip.txt

Processando: experimento5DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['5']
  ✅ experimento_R5_trajetoria_tail_tip.txt

Processando: experimento7DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['7']
  ✅ experimento_R7_trajetoria_tail_tip.txt

Processando: experimento13DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['13']
  ✅ experimento_R13_trajetoria_tail_tip.txt

Processando: experimento18DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['18']
  ✅ experimento_R18_trajetoria_tail_tip.txt

Processando: experimento16DLC_resnet50_validacao_topscan-dlcMay21shuffle1_120000_filtered.h5
ID do rato: ['16']
  ✅ experimento_R16_trajetoria_tail_tip.txt

Processando: experimento2DLC_resnet50_validacao_topscan-dlcMay2

##FUNÇÕES DE PRÉ-PROCESSAMENTO: OUTLIERS E SUAVIZAÇÃO

In [ ]:
def filtrar_outliers_trajetoria(df, col_x='X', col_y='Y', eps=15.0, min_samples=3, iqr_multiplicador=2.5):
    """
    Remove outliers por dois métodos complementares:
    1. IQR sobre o deslocamento frame-a-frame (saltos fisicamente impossíveis).
    2. DBSCAN sobre as coordenadas válidas restantes (glitches de predição isolados).

    Parâmetros para roedores em arenas (~40x40 cm, 30 fps):
      eps=15 px  → distância máxima entre pontos do cluster principal. Ajustar conforme px/cm do vídeo.
      iqr_multiplicador=2.5 → mais conservador que 1.5 para preservar acelerações reais de fuga.
    """
    df = df.copy()

    # --- Passo 1: IQR sobre deslocamento ---
    dx = np.diff(df[col_x].fillna(method='ffill').values, prepend=df[col_x].iloc[0])
    dy = np.diff(df[col_y].fillna(method='ffill').values, prepend=df[col_y].iloc[0])
    deslocamento = np.sqrt(dx**2 + dy**2)

    Q1, Q3  = np.nanpercentile(deslocamento, 25), np.nanpercentile(deslocamento, 75)
    limiar  = Q3 + iqr_multiplicador * (Q3 - Q1)
    saltos  = deslocamento > limiar
    df.loc[saltos, [col_x, col_y]] = np.nan
    print(f"  IQR: {saltos.sum()} saltos removidos (limiar: {limiar:.1f} px/frame)")

    # --- Passo 2: DBSCAN sobre coordenadas válidas ---
    idx_validos     = df[[col_x, col_y]].dropna().index
    coords_validos  = df.loc[idx_validos, [col_x, col_y]].values

    if len(coords_validos) > min_samples:
        labels          = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(coords_validos)
        cluster_princ   = pd.Series(labels[labels != -1]).mode()[0] if any(labels != -1) else 0
        idx_ruido       = idx_validos[labels != cluster_princ]
        df.loc[idx_ruido, [col_x, col_y]] = np.nan
        n_ruido = int((labels == -1).sum() + (labels != cluster_princ).sum())
        print(f"  DBSCAN: {n_ruido} pontos isolados removidos")

    return df


def suavizar_trajetoria(df, col_x='X', col_y='Y', janela=11, ordem=3):
    """
    Filtro Savitzky-Golay após interpolação linear dos NaN internos.

    janela=11 frames (~0.37 s a 30 fps): preserva acelerações de exploração e fuga.
    ordem=3: polinômio cúbico — captura curvatura sem achatar picos de velocidade.
    Janelas >21 frames tendem a distorcer paradas bruscas e viradas em U.
    """
    df = df.copy()
    for col in [col_x, col_y]:
        serie        = df[col].copy()
        serie_interp = serie.interpolate(method='linear', limit_direction='forward', limit=5)
        n_validos    = serie_interp.notna().sum()

        if n_validos >= janela:
            valores_sg = savgol_filter(
                serie_interp.fillna(method='ffill').fillna(method='bfill'),
                window_length=janela, polyorder=ordem,
            )
            # Restaurar NaN originais — não mascarar ausência de dado com suavização
            df[col] = np.where(serie.isna(), np.nan, valores_sg)
        else:
            print(f"  Aviso: {col} com apenas {n_validos} pontos válidos — SG não aplicado.")

    return df

##CRIAÇÃO DE TXT SENSÍVEL (TOPSCAN) COM CALIBRAÇÃO POR REGRESSÃO

In [ ]:
# --- 1. Configuração dos Diretórios ---
base_path        = "/content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT original e XLSX original/"
saida_path_dir   = "/content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/"
diretorio_dlc_ref = "/content/drive/MyDrive/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/"


# --- 2. Diagnóstico de Alinhamento Temporal ---
def diagnosticar_alinhamento_temporal(df_ts, df_dlc, col_frame='FrameNum'):
    """
    Verifica integridade temporal antes do merge:
    - Frames duplicados e gaps em cada série.
    - Percentual de sobreposição entre TS e DLC.
    - Estimativa de offset por correlação cruzada quando sobreposição < 85%.
    """
    print("\n  --- Diagnóstico Temporal ---")
    for nome, df in [('TopScan', df_ts), ('DLC', df_dlc)]:
        frames = df[col_frame].sort_values()
        n_dupl = int(frames.duplicated().sum())
        gaps   = frames.diff().dropna()
        n_gaps = int((gaps > 1).sum())
        if n_dupl:
            print(f"  {nome}: {n_dupl} frames DUPLICADOS")
        if n_gaps:
            print(f"  {nome}: {n_gaps} gaps na sequência de frames")
        if not n_dupl and not n_gaps:
            print(f"  {nome}: sequência contínua ({int(frames.min())}–{int(frames.max())})")

    frames_ts  = set(df_ts[col_frame])
    frames_dlc = set(df_dlc[col_frame])
    overlap    = len(frames_ts & frames_dlc)
    pct        = overlap / max(len(frames_ts), len(frames_dlc)) * 100
    print(f"  Sobreposição: {overlap} frames ({pct:.1f}%) | TS={len(frames_ts)}, DLC={len(frames_dlc)}")

    if pct < 85:
        print("  Aviso: sobreposição < 85% — investigar offset temporal.")
        ts_sorted  = df_ts.sort_values(col_frame).set_index(col_frame)
        dlc_sorted = df_dlc.sort_values(col_frame).set_index(col_frame)
        idx_comuns = ts_sorted.index.intersection(dlc_sorted.index)
        if len(idx_comuns) > 50:
            from numpy.fft import fft, ifft
            sinal_ts  = np.sqrt(ts_sorted.loc[idx_comuns, 'CenterX(mm)'].diff()**2 +
                                ts_sorted.loc[idx_comuns, 'CenterY(mm)'].diff()**2).fillna(0).values
            sinal_dlc = np.sqrt(dlc_sorted.loc[idx_comuns, 'X_dlc'].diff()**2 +
                                dlc_sorted.loc[idx_comuns, 'Y_dlc'].diff()**2).fillna(0).values
            correlacao     = np.real(ifft(fft(sinal_ts) * np.conj(fft(sinal_dlc))))
            offset_estimado = int(np.argmax(correlacao[:len(correlacao) // 2]))
            print(f"  Offset estimado por cross-correlação: ~{offset_estimado} frames")

    return overlap


# --- 3. Calibração por Regressão Linear Robusta (Huber) ---
def calibrar_por_regressao(df_limpo_ts, df_dlc_ref,
                            col_ts_x='CenterX(mm)', col_ts_y='CenterY(mm)'):
    """
    Mapeia coordenadas TopScan -> espaço DLC via regressão Huber (robusta a outliers).
    Inclui intercepto — corrige translação além da escala.
    R² < 0.90 indica distorção de lente ou desalinhamento câmera/arena não-linear.
    """
    df_merged = pd.merge(df_limpo_ts, df_dlc_ref, on='FrameNum', how='inner').dropna()

    if len(df_merged) < 30:
        print("  Aviso: pontos insuficientes para regressão. Pulando calibração.")
        return df_limpo_ts.copy(), None

    diagnosticar_alinhamento_temporal(df_limpo_ts, df_dlc_ref)

    df_calibrado = df_limpo_ts.copy()
    modelos      = {}

    for eixo_ts, eixo_dlc in [(col_ts_x, 'X_dlc'), (col_ts_y, 'Y_dlc')]:
        X      = df_merged[[eixo_ts]].values
        y_alvo = df_merged[eixo_dlc].values

        modelo = HuberRegressor(epsilon=1.5).fit(X, y_alvo)
        r2     = r2_score(y_alvo, modelo.predict(X))

        print(f"  Regressão {eixo_ts}: coef={modelo.coef_[0]:.5f}, intercept={modelo.intercept_:.3f}, R²={r2:.4f}")
        if r2 < 0.90:
            print(f"  Aviso: R²={r2:.3f} baixo em {eixo_ts}. Verificar alinhamento câmera/arena.")

        df_calibrado[eixo_ts] = modelo.coef_[0] * df_calibrado[eixo_ts] + modelo.intercept_
        modelos[eixo_ts] = {'coef': modelo.coef_[0], 'intercept': modelo.intercept_, 'r2': r2}

    return df_calibrado, modelos


# --- 4. Função Principal de Processamento ---
def processar_e_calibrar_topscan(txt_path, xlsx_path, saida_path):
    print(f"\nIniciando processo para {os.path.basename(txt_path)}...")

    # Leitura do TXT original do TopScan
    try:
        colunas_txt      = ['FrameNum', 'CenterX(mm)', 'CenterY(mm)', 'Areas']
        df_trajetoria_ts = pd.read_csv(txt_path, sep=r'\s+', skiprows=1, names=colunas_txt)
        df_trajetoria_ts['Areas'] = df_trajetoria_ts['Areas'].str.replace(',', '', regex=False)
    except Exception as e:
        print(f"  ERRO ao ler TXT do TopScan: {e}")
        return

    df_trajetoria_ts['CenterX(mm)'] = pd.to_numeric(df_trajetoria_ts['CenterX(mm)'], errors='coerce')
    df_trajetoria_ts['CenterY(mm)'] = pd.to_numeric(df_trajetoria_ts['CenterY(mm)'], errors='coerce')

    # Remover flag de "animal não detectado" (-1) do TopScan
    df_limpo_ts = df_trajetoria_ts[
        (df_trajetoria_ts['CenterX(mm)'] != -1) & (df_trajetoria_ts['CenterY(mm)'] != -1)
    ].copy()
    print(f"  TopScan: {len(df_limpo_ts)} linhas após remoção de -1.")

    # Filtrar outliers (IQR + DBSCAN)
    df_limpo_ts = filtrar_outliers_trajetoria(df_limpo_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')

    # Suavização cinemática (Savitzky-Golay)
    df_limpo_ts = suavizar_trajetoria(df_limpo_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')

    # Calibração por regressão
    match = re.search(r'RATO_(\d+)', os.path.basename(txt_path))
    if not match:
        print("  Aviso: ID do rato não encontrado no nome do arquivo. Pulando calibração.")
        df_calibrado_ts = df_limpo_ts.copy()
    else:
        rat_id        = match.group(1)
        path_dlc_ref  = os.path.join(diretorio_dlc_ref, f"experimento_R{rat_id}_trajetoria_centerbody.txt")

        if not os.path.exists(path_dlc_ref):
            print(f"  Aviso: referência DLC não encontrada em '{path_dlc_ref}'. Pulando calibração.")
            df_calibrado_ts = df_limpo_ts.copy()
        else:
            try:
                df_dlc_ref = pd.read_csv(path_dlc_ref, sep=r'\s+').rename(
                    columns={'Frame': 'FrameNum', 'X': 'X_dlc', 'Y': 'Y_dlc'}
                )
                # Remover frames DLC com NaN (likelihood < limiar exportados como NaN)
                df_dlc_ref = df_dlc_ref.dropna(subset=['X_dlc', 'Y_dlc'])

                df_calibrado_ts, _ = calibrar_por_regressao(df_limpo_ts, df_dlc_ref)
            except Exception as e:
                print(f"  ERRO durante calibração: {e}")
                df_calibrado_ts = df_limpo_ts.copy()

    # Leitura do Excel para eventos
    try:
        df_eventos = pd.read_excel(xlsx_path, header=1)
        df_eventos.columns = df_eventos.columns.str.strip()
    except Exception as e:
        print(f"  ERRO ao ler Excel: {e}")
        return

    # Atribuição de eventos à coluna Areas
    df_mesclado_final = df_calibrado_ts.copy()
    df_mesclado_final['Areas'] = 'Floor'

    for _, row in df_eventos.iterrows():
        try:
            frame_inicio = int(row['From Frame'])
            frame_fim    = int(row['To Frame'])
            evento       = str(row['Event'])
            evento_limpo = evento.split("sniffing On")[-1].strip() if "sniffing On" in evento else evento.strip()
            df_mesclado_final.loc[
                (df_mesclado_final['FrameNum'] >= frame_inicio) &
                (df_mesclado_final['FrameNum'] <= frame_fim), 'Areas'
            ] = evento_limpo
        except (KeyError, ValueError) as e:
            print(f"  Aviso: problema ao processar linha do Excel: {e}. Pulando.")
            continue

    # Salvar TXT final
    os.makedirs(os.path.dirname(saida_path), exist_ok=True)
    with open(saida_path, 'w') as f_out:
        f_out.write("FrameNum CenterX(mm) CenterY(mm) Areas\n")
        for _, row in df_mesclado_final.iterrows():
            f_out.write(f"{int(row['FrameNum'])} {row['CenterX(mm)']:.2f} {row['CenterY(mm)']:.2f} {row['Areas']}\n")
    print(f"  Arquivo salvo em: {saida_path}")


# --- 5. Execução Automática ---
def executar_processo_automatico():
    print("--- Iniciando Calibração e Mesclagem do TopScan ---")
    if not os.path.isdir(base_path):
        print(f"ERRO: diretório base '{base_path}' não encontrado.")
        return

    nomes_base = sorted({os.path.splitext(f)[0] for f in os.listdir(base_path) if f.upper().endswith('.TXT')})

    for nome_base in nomes_base:
        txt_path_atual  = os.path.join(base_path, f"{nome_base}.TXT")
        xlsx_path_atual = os.path.join(base_path, f"{nome_base}.xlsx")
        saida_path_atual = os.path.join(saida_path_dir, f"{nome_base}_NOVO.TXT")

        if os.path.exists(txt_path_atual) and os.path.exists(xlsx_path_atual):
            processar_e_calibrar_topscan(txt_path_atual, xlsx_path_atual, saida_path_atual)
        else:
            print(f"  Aviso: TXT ou XLSX ausente para '{nome_base}'. Pulando.")

    print("\n--- Processo finalizado ---")

executar_processo_automatico()

--- Iniciando Processo Automático de Calibração e Mesclagem do TopScan ---

--- Processando: RATO_10_TCR ---
Iniciando processo para RATO_10_TCR.TXT...
  Dados de trajetória TopScan limpos: 8936 linhas válidas encontradas.
  Fator de Escala X (TS/DLC) calculado: 4.8437
  Fator de Escala Y (TS/DLC) calculado: 3.8533
  Calibração aplicada com sucesso ao TopScan.
  ✅ Arquivo calibrado e mesclado salvo em: /content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/RATO_10_TCR_NOVO.TXT

--- Processando: RATO_12_TCR ---
Iniciando processo para RATO_12_TCR.TXT...
  Dados de trajetória TopScan limpos: 8971 linhas válidas encontradas.
  Fator de Escala X (TS/DLC) calculado: 5.2125
  Fator de Escala Y (TS/DLC) calculado: 11.6934
  Calibração aplicada com sucesso ao TopScan.
  ✅ Arquivo calibrado e mesclado salvo em: /content/drive/MyDrive/validacao_topscan/TOPSCAN/TXT novo/RATO_12_TCR_NOVO.TXT

--- Processando: RATO_15_TCR ---
Iniciando processo para RATO_15_TCR.TXT...
  Dados de trajetória TopSc